In [2]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-180e40d8-37b4-a91a-c89f-18aab2e67fdd)
GPU 1: Tesla T4 (UUID: GPU-9f0c6e1c-5384-83b3-fa64-403801d6429b)


In [29]:
!rm -rf build
!cmake -B build -DGGML_CUDA=OFF
!cmake --build build --config Release -j2

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 0.1.0-dev
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found Ope

In [ ]:
# 1. Instalare rapidă zstd, Ollama și Cloudflared
!apt-get update -y && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# 2. Pornire Ollama cu GPU & Tunel Cloudflare
import subprocess, os, time, re

env = os.environ.copy()
env["OLLAMA_ORIGINS"] = "*"
env["OLLAMA_HOST"] = "0.0.0.0:11434"
env["CUDA_VISIBLE_DEVICES"] = "0,1"

subprocess.Popen(["/usr/local/bin/ollama", "serve"], env=env, stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(3)

subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:11434"], stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)

# 3. Verificare conexiune & Afișare URL
import urllib.request
try:
    with urllib.request.urlopen("http://127.0.0.1:11434/") as res:
        print("✅ OLLAMA STATUS:", res.read().decode().strip())
except Exception as e:
    print("❌ Eroare pornire:", e)

with open("tunnel.log", "r") as f:
    logs = f.read()
    urls = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', logs)
    if urls:
        print("\n" + "="*60)
        print("🚀 URL TUNEL ACTIV & CONECTAT LA GPU:")
        print(urls[0])
        print("="*60 + "\n")